# Packages and Functions

## Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import os
from pathlib import Path

## Functions

In [ ]:
def response_to_tox_class(response_value):
    """Convert Response value to Tox toxicity class"""
    if response_value <= 5:
        return 0  # Tox Level 0
    elif response_value <= 50:
        return 1  # Tox Level 1
    elif response_value <= 500:
        return 2  # Tox Level 2
    elif response_value <= 5000:
        return 3  # Tox Level 3
    else:
        return 4  # Tox Level 4

def load_classification_outputs(base_folder, folder_name, bin_val, threshold_val, use_super_test=False):
    """
    Load classification outputs from the saved folder structure
    """
    # Construct folder path - the outputs are in cond_enc_1234e1e2_classification_df6
    folder_name = folder_name
    folder_path = os.path.join(base_folder, folder_name)
    
    # Convert bin_val and threshold_val to the naming format (replace . with _)
    bin_part = str(bin_val).replace('.', '_')
    threshold_part = str(threshold_val).replace('.', '_')
    
    # Choose file name pattern based on use_super_test parameter
    if use_super_test:
        filename = f"super_test_cond_enc_bin{bin_part}_thresh{threshold_part}_df_spectra.parquet"
    else:
        filename = f"cond_enc_bin{bin_part}_thresh{threshold_part}_df_spectra.parquet"
    
    outputs_file = os.path.join(folder_path, filename)
    
    if not os.path.exists(outputs_file):
        raise FileNotFoundError(f"Output file not found: {outputs_file}")
    
    df = pd.read_parquet(outputs_file)
    
    # Add Tox class from Response values
    df['tox_class'] = df['Response'].apply(response_to_tox_class)
    
    return df

def create_confusion_matrix_plot(y_true, y_pred, class_labels=None, title="Confusion Matrix", save_path=None):
    """
    Create and display a confusion matrix plot
    """
    # Create confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Set up class labels
    if class_labels is None:
        class_labels = [f"Class {i}" for i in range(len(np.unique(y_true)))]
    
    # Create the plot
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_labels, yticklabels=class_labels)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()
    
    return cm

def analyze_classification_results(base_folder, bin_val, threshold_val, use_super_test=False, save_plots=True):
    """
    Complete analysis function that loads outputs and creates confusion matrices
    """
    # Load the outputs
    try:
        df = load_classification_outputs(base_folder, bin_val, threshold_val, use_super_test)
        test_type = "Super Test" if use_super_test else "Regular Test"
        print(f"Loaded {test_type} outputs for bin={bin_val}, threshold={threshold_val}")
        print(f"Dataset shape: {df.shape}")
    except FileNotFoundError as e:
        print(f"Error: {e}")
        return None, None
    
    # Define Tox toxicity class labels
    tox_labels = ["Tox Level 0", "Tox Level 1", "Tox Level 2", "Tox Level 3", "Tox Level 4"]
    
    # Extract true and predicted labels
    y_true = df['tox_class'].values  # Tox class derived from Response
    y_pred = df['cond_tox_pred_class'].values  # Predicted class
    
    print(f"Using Tox classes derived from Response values")
    
    # Since true labels are 0,1,2,3,4 and predicted are 0,1,2,3,4 we need to adjust
    print(f"True label distribution: {np.bincount(y_true, minlength=5)}")
    print(f"Predicted label distribution: {np.bincount(y_pred, minlength=5)}")
    
    # Convert true labels from 1,2,3,4 to 0,1,2,3 to match predictions
    y_true_adjusted = y_true
    
    # Create confusion matrix
    test_label = "Super Test" if use_super_test else "Regular Test"
    cm = create_confusion_matrix_plot(
        y_true_adjusted, y_pred, 
        class_labels=tox_labels,
        title=f"Confusion Matrix - {test_label} (Bin={bin_val}, Threshold={threshold_val})",
        save_path=f"confusion_matrix_{'super_test_' if use_super_test else ''}bin{bin_val}_thr{threshold_val}.png" if save_plots else None
    )
    
    # Print classification report
    print("\nClassification Report:")
    print(classification_report(y_true_adjusted, y_pred, target_names=tox_labels))
    
    # Calculate and print accuracy metrics
    accuracy = np.sum(y_true_adjusted == y_pred) / len(y_true_adjusted)
    print(f"\nOverall Accuracy: {accuracy:.4f}")
    
    # Per-class accuracy
    print("\nPer-class Accuracy:")
    for i, label in enumerate(tox_labels):
        class_mask = y_true_adjusted == i
        if np.sum(class_mask) > 0:
            class_acc = np.sum((y_true_adjusted == y_pred) & class_mask) / np.sum(class_mask)
            print(f"{label}: {class_acc:.4f}")
    
    return df, cm

def compare_multiple_configurations(base_folder, bin_threshold_pairs, use_super_test=False, save_plots=True):
    """
    Compare results across multiple bin/threshold configurations
    """
    results = {}
    
    for bin_val, threshold_val in bin_threshold_pairs:
        print(f"\n{'='*50}")
        print(f"Analyzing Bin={bin_val}, Threshold={threshold_val}")
        print(f"{'='*50}")
        
        try:
            df, cm = analyze_classification_results(base_folder, bin_val, threshold_val, use_super_test, save_plots)
            if df is not None:
                y_true = df['tox_class'].values
                y_pred = df['cond_tox_pred_class'].values
                accuracy = np.sum(y_true == y_pred) / len(y_true)
                
                results[(bin_val, threshold_val)] = {
                    'accuracy': accuracy,
                    'confusion_matrix': cm,
                    'dataframe': df
                }
                
        except Exception as e:
            print(f"Error processing bin={bin_val}, threshold={threshold_val}: {e}")
    
    # Summary comparison
    if results:
        print(f"\n{'='*60}")
        print("SUMMARY COMPARISON")
        print(f"{'='*60}")
        print(f"{'Bin':<8} {'Threshold':<12} {'Accuracy':<10}")
        print("-" * 30)
        
        for (bin_val, threshold_val), metrics in results.items():
            print(f"{bin_val:<8} {threshold_val:<12} {metrics['accuracy']:.4f}")
    
    return results

def visualize_classification(folder, bin_size, threshold_size, use_super_test=False, save_plots=True):
    """
    Main visualization function that takes folder, bin size, and threshold size
    
    Parameters:
    folder (str): Base folder path containing the classification outputs
    bin_size (int/float): Bin size value
    threshold_size (float): Threshold size value  
    use_super_test (bool): Whether to load super test results or regular test results
    save_plots (bool): Whether to save plots as PNG files
    
    Returns:
    tuple: DataFrame and confusion matrix from the analysis
    """
    test_type = "Super Test" if use_super_test else "Regular Test"
    print(f"Analyzing {test_type} classification results:")
    print(f"Folder: {folder}")
    print(f"Bin size: {bin_size}")
    print(f"Threshold size: {threshold_size}")
    print("-" * 50)
    
    # Run the analysis
    df, cm = analyze_classification_results(folder, bin_size, threshold_size, use_super_test, save_plots)
    
    return df, cm

In [ ]:
# Create percentage-based confusion matrix heatmap
def create_percentage_confusion_matrix_standalone(folder, folder_name, bin_size, threshold_size, use_super_test=False, save_plots=True, test_only=True):
    """
    Standalone function to create percentage confusion matrix from classification outputs
    Each row shows how the actual label was distributed across predictions
    
    Parameters:
    test_only (bool): If True, only include test/validation spectra (exclude training spectra where train=1)
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import confusion_matrix
    import os
    
    def response_to_tox_class_local(response_value):
        """Convert Response value to Tox toxicity class"""
        if response_value <= 5:
            return 0  # Tox Level 0
        elif response_value <= 50:
            return 1  # Tox Level 1
        elif response_value <= 500:
            return 2  # Tox Level 2
        elif response_value <= 5000:
            return 3  # Tox Level 3
        else:
            return 4  # Tox Level 4
    
    # Load the classification outputs
    folder_name = folder_name
    folder_path = os.path.join(folder, folder_name)
    
    # Convert bin_size and threshold_size to the naming format (replace . with _)
    bin_part = str(bin_size).replace('.', '_')
    threshold_part = str(threshold_size).replace('.', '_')
    
    # Choose file name pattern based on use_super_test parameter
    if use_super_test:
        filename = f"super_test_cond_enc_bin{bin_part}_thresh{threshold_part}_df_spectra.parquet"
    else:
        filename = f"cond_enc_bin{bin_part}_thresh{threshold_part}_df_spectra.parquet"
    
    outputs_file = os.path.join(folder_path, filename)
    
    if not os.path.exists(outputs_file):
        raise FileNotFoundError(f"Output file not found: {outputs_file}")
    
    df = pd.read_parquet(outputs_file)
    
    # Filter to test-only spectra if requested
    if test_only:
        if 'train' in df.columns:
            original_count = len(df)
            df = df[df['train'] == 0].copy()
            filtered_count = len(df)
            print(f"Filtered to test-only spectra: {filtered_count}/{original_count} samples ({filtered_count/original_count*100:.1f}%)")
        else:
            print("Warning: 'train' column not found, cannot filter to test-only spectra")
    
    # Add Tox class from Response values
    df['tox_class'] = df['Response'].apply(response_to_tox_class_local)
    
    # Extract and adjust labels
    y_true = df['tox_class'].values
    y_pred = df['cond_tox_pred_class'].values
    
    # Define Tox labels
    class_labels = ["Tox Level 0", "Tox Level 1", "Tox Level 2", "Tox Level 3", "Tox Level 4"]
    
    # Create confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])
    
    # Convert to percentages (row-wise normalization)
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Create the plot
    plt.figure(figsize=(10, 8))
    
    # Create heatmap with percentage values
    sns.heatmap(cm_percent, 
                annot=True, 
                fmt='.1f', 
                cmap='Blues', 
                xticklabels=class_labels, 
                yticklabels=class_labels,
                cbar_kws={'label': 'Percentage (%)'},
                square=True)
    
    test_type = "Super Test" if use_super_test else "Regular Test"
    test_label = " (Test Only)" if test_only else ""
    plt.title(f"Confusion Matrix - {test_type} Percentages{test_label} (Bin={bin_size}, Threshold={threshold_size})")
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    
    if save_plots:
        save_path = f"confusion_matrix_percentages_{'super_test_' if use_super_test else ''}bin{bin_size}_thr{threshold_size}.png"
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved as: {save_path}")
    
    plt.show()
    
    # Print percentage breakdown with proportions
    print(f"\nPercentage breakdown by actual label (Bin={bin_size}, Threshold={threshold_size}):")
    for i, actual_label in enumerate(class_labels):
        row_total = cm.sum(axis=1)[i]
        print(f"\n{actual_label} (Total: {row_total} samples):")
        for j, pred_label in enumerate(class_labels):
            count = cm[i, j]
            percentage = cm_percent[i, j]
            print(f"  → {pred_label}: {percentage:.1f}% ({count}/{row_total} samples)")
    
    return cm_percent, df

# Conditional Encoder 1234e1e2 Classifier

## Confusion Matricies

In [ ]:
# Regular Test Percentage Confusion Matrix
cm_percent, df_analysis = create_percentage_confusion_matrix_standalone(
    folder="/home/dlipsey/MITLincolnLabs/MIT_LL_data/",
    folder_name="cond_enc_1234e1e2_classification_df6",
    bin_size=1,
    threshold_size=0.05,
    use_super_test=False,
    save_plots=False,
    test_only=False
)

In [ ]:
# Regular Test Percentage Confusion Matrix
cm_percent, df_analysis = create_percentage_confusion_matrix_standalone(
    folder="/home/dlipsey/MITLincolnLabs/MIT_LL_data/",
    folder_name="cond_enc_134e1e2_2stepclassi_df6",
    bin_size=1,
    threshold_size=0.05,
    use_super_test=False,
    save_plots=False,
    test_only=True
)

In [ ]:
# Super Test Percentage Confusion Matrix
cm_percent, df_analysis = create_percentage_confusion_matrix_standalone(
    folder="/home/dlipsey/MITLincolnLabs/MIT_LL_data/",
    folder_name="cond_enc_1234e1e2_classification_df6_super_test",
    bin_size=1,
    threshold_size=0.05,
    use_super_test=True,
    save_plots=False
)

## Heatmap

### Heatmap Function

In [ ]:
# Modified Configurable Accuracy Heatmap Function with Super Test Support
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, accuracy_score

def response_to_tox_class_local_v2(response_value):
    """Convert Response value to Tox toxicity class"""
    if response_value <= 5:
        return 0  # Tox Level 0
    elif response_value <= 50:
        return 1  # Tox Level 1
    elif response_value <= 500:
        return 2  # Tox Level 2
    elif response_value <= 5000:
        return 3  # Tox Level 3
    else:
        return 4  # Tox Level 4

def calculate_class_accuracy_v2(y_true, y_pred, target_class):
    """Calculate accuracy for a specific class"""
    # Convert to 0-based indexing
    target_class_idx = target_class - 1
    
    # Get mask for samples that actually belong to target class
    class_mask = y_true == target_class_idx
    
    if np.sum(class_mask) == 0:
        return np.nan  # No samples of this class
    
    # Calculate accuracy for this class (correct predictions / total actual samples of this class)
    correct_for_class = np.sum((y_true == y_pred) & class_mask)
    total_for_class = np.sum(class_mask)
    
    return correct_for_class / total_for_class

def parse_cond_enc_classification_dataset_name_v2(dataset_name, use_super_test=False):
    """Extract bin size and threshold from conditional encoder classification dataset name"""
    # Remove 'super_test_' prefix if present
    if use_super_test and dataset_name.startswith('super_test_'):
        name_part = dataset_name.replace('super_test_cond_enc_', '')
    else:
        # Remove 'cond_enc_' prefix
        name_part = dataset_name.replace('cond_enc_', '')
    
    # Handle thresh_zero case (no threshold)
    if 'thresh_zero' in name_part:
        # Extract bin size
        bin_part = name_part.split('_thresh_zero')[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        threshold = 0.0
    else:
        # Extract bin size and threshold
        parts = name_part.split('_thresh')
        bin_part = parts[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        
        thresh_part = parts[1].split('_df_spectra')[0]
        threshold = float(thresh_part.replace('_', '.'))
    
    return bin_size, threshold

def load_and_process_classification_data_v2(folder_path, use_super_test=False, test_only=True):
    """Load and process classification data from the specified folder
    
    Parameters:
    test_only (bool): If True, only include test/validation spectra (exclude training spectra where train=1)
    """
    # Check if folder exists
    if not os.path.exists(folder_path):
        print(f"Error: Folder {folder_path} does not exist!")
        return pd.DataFrame()
    
    # Get all .parquet files in the folder
    parquet_files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]
    
    # Filter files based on whether we want super test or regular test
    if use_super_test:
        # Only include files that start with 'super_test_'
        dataset_names = [f.replace('.parquet', '') for f in parquet_files if f.startswith('super_test_')]
    else:
        # Only include files that start with 'cond_enc_' but NOT 'super_test_'
        dataset_names = [f.replace('.parquet', '') for f in parquet_files 
                        if f.startswith('cond_enc_') and not f.startswith('super_test_')]

    if not dataset_names:
        test_type = "super test" if use_super_test else "regular"
        print(f"Warning: No {test_type} .parquet files found in {folder_path}")
        return pd.DataFrame()

    # Initialize storage for classification results
    classification_results = []

    # Process classification datasets
    for i, dataset_name in enumerate(sorted(dataset_names), 1):
        try:
            # Load the classification dataset
            file_path = os.path.join(folder_path, f"{dataset_name}.parquet")
            df = pd.read_parquet(file_path)
            
            # Filter to test-only spectra if requested
            if test_only:
                if 'train' in df.columns:
                    original_count = len(df)
                    df = df[df['train'] == 0].copy()
                    filtered_count = len(df)
                    print(f"  {dataset_name}: Filtered to test-only - {filtered_count}/{original_count} samples")
                else:
                    print(f"  {dataset_name}: Warning - 'train' column not found, cannot filter to test-only spectra")
            
            # Check if required columns exist
            if 'cond_tox_pred_class' not in df.columns:
                print(f"  Warning: 'cond_tox_pred_class' column not found, skipping...")
                continue
            
            if 'Response' not in df.columns:
                print(f"  Warning: 'Response' column not found, skipping...")
                continue
            
            # Get predictions and true values
            y_pred_class = df['cond_tox_pred_class']  # Predicted Tox classes
            y_true_response = df['Response']  # True toxicity values
            
            # Convert true response values to Tox classes
            y_true_class = y_true_response.apply(response_to_tox_class_local_v2)
            
            # Remove rows with NaN values
            valid_mask = ~(y_pred_class.isna() | y_true_class.isna())
            y_pred_class_clean = y_pred_class[valid_mask]
            y_true_class_clean = y_true_class[valid_mask]
            
            if len(y_pred_class_clean) < 10:  # Skip if too few samples
                print(f"  Too few samples, skipping...")
                continue
            
            # Convert to 0-based indexing for sklearn functions
            y_true_class_clean_0based = y_true_class_clean - 1
            
            # Calculate overall accuracy
            overall_accuracy = accuracy_score(y_true_class_clean_0based, y_pred_class_clean)
            
            # Calculate individual class accuracies
            class1_accuracy = calculate_class_accuracy_v2(y_true_class_clean_0based, y_pred_class_clean, 1)
            class2_accuracy = calculate_class_accuracy_v2(y_true_class_clean_0based, y_pred_class_clean, 2)
            class3_accuracy = calculate_class_accuracy_v2(y_true_class_clean_0based, y_pred_class_clean, 3)
            class4_accuracy = calculate_class_accuracy_v2(y_true_class_clean_0based, y_pred_class_clean, 4)
            
            # Store results
            classification_results.append({
                'Dataset': dataset_name,
                'Overall_Accuracy': overall_accuracy,
                'Class1_Accuracy': class1_accuracy,
                'Class2_Accuracy': class2_accuracy,
                'Class3_Accuracy': class3_accuracy,
                'Class4_Accuracy': class4_accuracy,
                'Samples': len(y_pred_class_clean)
            })
            
        except Exception as e:
            print(f"  ✗ Error processing {dataset_name}: {str(e)}")
            continue

    # Convert results to DataFrame
    results_df = pd.DataFrame(classification_results)

    if results_df.empty:
        print("Warning: No datasets were successfully processed!")
        return results_df

    # Add bin_size and threshold columns
    bin_sizes = []
    thresholds = []

    for dataset_name in results_df['Dataset']:
        bin_size, threshold = parse_cond_enc_classification_dataset_name_v2(dataset_name, use_super_test)
        bin_sizes.append(bin_size)
        thresholds.append(threshold)

    results_df['BinSize'] = bin_sizes
    results_df['Threshold'] = thresholds

    # Remove duplicates
    results_df = results_df.drop_duplicates(subset=['BinSize', 'Threshold'], keep='first')
    
    return results_df

# Main function to create accuracy heatmap with configurable parameters and super test support
def create_detailed_heatmap_cond_enc_accuracy_v2(folder_path, metric='average', colormap='RdYlGn', figsize=(12, 8), use_super_test=False, test_only=True):
    """
    Create a detailed heatmap for Classification Accuracy with support for super test datasets
    
    Parameters:
    folder_path (str): Path to folder containing classification result files
    metric (str): Type of accuracy to plot
        - 'average': Overall accuracy
        - '1', '2', '3', '4': Individual class accuracy (Tox Level 0-4)
    colormap (str): Color scheme for the heatmap (e.g., 'RdYlGn', 'viridis', 'Blues', 'Reds', 'Greens', 'Purples')
    figsize (tuple): Figure size as (width, height)
    use_super_test (bool): Whether to load super test datasets (files starting with 'super_test_') or regular datasets
    test_only (bool): If True, only include test/validation spectra (exclude training spectra where train=1)
    
    Returns:
    pandas.DataFrame: Pivot table with accuracy data, or None if no data available
    """
    
    # Load and process data from the specified folder
    df_results = load_and_process_classification_data_v2(folder_path, use_super_test, test_only)
    
    if df_results.empty:
        test_type = "super test" if use_super_test else "regular"
        print(f"No {test_type} data available to create heatmap!")
        print("Please ensure the folder contains valid .parquet files with required columns.")
        return None
    
    # Map metric types to column names and display names
    metric_mapping = {
        'average': ('Overall_Accuracy', 'Overall Accuracy'),
        '0': ('Class0_Accuracy', 'Tox Level 0 Accuracy'),
        '1': ('Class1_Accuracy', 'Tox Level 1 Accuracy'),
        '2': ('Class2_Accuracy', 'Tox Level 2 Accuracy'),
        '3': ('Class3_Accuracy', 'Tox Level 3 Accuracy'),
        '4': ('Class4_Accuracy', 'Tox Level 4 Accuracy')
    }
    
    if metric not in metric_mapping:
        raise ValueError(f"metric must be one of: {list(metric_mapping.keys())}")
    
    column_name, display_name = metric_mapping[metric]
    
    # Create pivot table
    accuracy_pivot = df_results.pivot(index='BinSize', columns='Threshold', values=column_name)
    
    # List all expected thresholds and bin sizes
    thresholds_subset = [0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 50, 100]
    bins_subset = [0.1, 0.5, 1, 2, 5, 10, 25, 50, 100, 200, 500]
    
    # Reindex pivot table
    accuracy_pivot = accuracy_pivot.reindex(columns=thresholds_subset, index=bins_subset)
    
    plt.figure(figsize=figsize)
    
    # Create heatmap
    ax = sns.heatmap(accuracy_pivot, 
                     annot=True, 
                     fmt='.3f', 
                     cmap=colormap,
                     square=False,
                     linewidths=0.5,
                     vmin=0,
                     vmax=1,
                     cbar_kws={'label': display_name, 'shrink': 0.8})
    
    # Add test type to title
    test_type = "Super Test" if use_super_test else "Regular Test"
    test_label = " (Test Only)" if test_only else ""
    plt.title(f'Classification Results ({test_type}): {display_name}{test_label} by Bin Size and Threshold', 
              fontsize=16, fontweight='bold')
    plt.xlabel('Threshold Value', fontsize=14)
    plt.ylabel('Bin Size', fontsize=14)
    plt.gca().invert_yaxis()
    
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    
    # Add text annotation for best performance
    best_acc = accuracy_pivot.max().max()
    plt.text(0.02, 0.98, f'Best {display_name}: {best_acc:.3f}', 
            transform=plt.gca().transAxes, 
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
            verticalalignment='top')
    
    plt.tight_layout()
    plt.show()
    
    return accuracy_pivot

### Heatmap Presentation

In [ ]:
# Regular test datasets
folder_path = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/cond_enc_1234e1e2_classification_df6"
create_detailed_heatmap_cond_enc_accuracy_v2(folder_path, metric='average', use_super_test=False, test_only=False)


In [ ]:
# Super test datasets  
folder_path = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/cond_enc_1234e1e2_classification_df6_super_test"
create_detailed_heatmap_cond_enc_accuracy_v2(folder_path, metric='average', use_super_test=True, test_only=True)

# Ablation Conditional Encoder 1234e1e2 Classifier

## Confusion Matricies

In [ ]:
# Regular Test Percentage Confusion Matrix
cm_percent, df_analysis = create_percentage_confusion_matrix_standalone(
    folder="/home/dlipsey/MITLincolnLabs/MIT_LL_data/",
    folder_name="ablation_classifier_1234e1e2_df6",
    bin_size=1,
    threshold_size=0.05,
    use_super_test=False,
    save_plots=False,
    test_only=True
)

In [ ]:
# Super Test Percentage Confusion Matrix
cm_percent, df_analysis = create_percentage_confusion_matrix_standalone(
    folder="/home/dlipsey/MITLincolnLabs/MIT_LL_data/",
    folder_name="ablation_classifier_1234e1e2_df6_super_test",
    bin_size=1,
    threshold_size=0.05,
    use_super_test=True,
    save_plots=False
)

## Heatmap

### Heatmap Function

In [ ]:
# In the previous cell

### Heatmap Creation

In [ ]:
# Regular test datasets
folder_path = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/ablation_classifier_1234e1e2_df6"
create_detailed_heatmap_cond_enc_accuracy_v2(folder_path, metric='1', use_super_test=False, test_only=True)

In [ ]:
# Super test datasets  
folder_path = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/ablation_classifier_1234e1e2_df6_super_test"
create_detailed_heatmap_cond_enc_accuracy_v2(folder_path, metric='1', use_super_test=True, test_only=True)

# Base Random Forest Classifier

## Confusion Matricies

In [ ]:
# Create percentage-based confusion matrix heatmap for Random Forest results
def create_percentage_confusion_matrix_random_forest(folder, folder_name, bin_size, threshold_size, save_plots=True, test_only=True):
    """
    Standalone function to create percentage confusion matrix from random forest classification outputs
    Each row shows how the actual label was distributed across predictions
    
    Parameters:
    test_only (bool): If True, only include test/validation spectra (exclude training spectra where train=1)
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import confusion_matrix
    import os
    
    def response_to_tox_class_local(response_value):
        """Convert Response value to Tox toxicity class"""
        if response_value <= 5:
            return 0  # Tox Level 0
        elif response_value <= 50:
            return 1  # Tox Level 1
        elif response_value <= 500:
            return 2  # Tox Level 2
        elif response_value <= 5000:
            return 3  # Tox Level 3
        else:
            return 4  # Tox Level 4
    
    # Load the random forest classification outputs
    folder_path = os.path.join(folder, folder_name)
    
    # Convert bin_size and threshold_size to the naming format (replace . with _)
    bin_part = str(bin_size).replace('.', '_')
    threshold_part = str(threshold_size).replace('.', '_')
    
    # Random forest file naming pattern
    filename = f"rf_bin{bin_part}_thresh{threshold_part}_df_spectra.parquet"
    
    outputs_file = os.path.join(folder_path, filename)
    
    if not os.path.exists(outputs_file):
        raise FileNotFoundError(f"Output file not found: {outputs_file}")
    
    df = pd.read_parquet(outputs_file)
    
    # Filter to test-only spectra if requested
    if test_only:
        if 'train' in df.columns:
            original_count = len(df)
            df = df[df['train'] == 0].copy()
            filtered_count = len(df)
            print(f"Filtered to test-only spectra: {filtered_count}/{original_count} samples ({filtered_count/original_count*100:.1f}%)")
        else:
            print("Warning: 'train' column not found, cannot filter to test-only spectra")
    
    # Get true labels - prioritize Tox_level if available, otherwise derive from Response
    if 'Tox_level' in df.columns:
        y_true_raw = df['Tox_level'].values  
    else:
        # Derive Tox class from Response values
        df['Tox_class'] = df['Response'].apply(response_to_tox_class_local)
        y_true_raw = df['Tox_class'].values  
    
    # Get predicted classes from random forest
    y_pred_raw = df['rf_predicted_tox_class'].values  
    
    # Convert both to 0-based indexing for sklearn
    y_true = y_true_raw   
    y_pred = y_pred_raw   
    
    # Define Tox labels
    class_labels = ["Tox Level 0","Tox Level 1", "Tox Level 2", "Tox Level 3", "Tox Level 4"]
    
    # Create confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])
    
    # Convert to percentages (row-wise normalization)
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Create the plot
    plt.figure(figsize=(10, 8))
    
    # Create heatmap with percentage values
    sns.heatmap(cm_percent, 
                annot=True, 
                fmt='.1f', 
                cmap='Blues', 
                xticklabels=class_labels, 
                yticklabels=class_labels,
                cbar_kws={'label': 'Percentage (%)'},
                square=True)
    
    test_label = " (Test Only)" if test_only else ""
    plt.title(f"Random Forest Confusion Matrix - Percentages{test_label}\n(Bin={bin_size}, Threshold={threshold_size})", 
              fontsize=14, fontweight='bold')
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('Actual', fontsize=12)
    plt.tight_layout()
    
    if save_plots:
        save_path = f"confusion_matrix_percentages_rf_bin{bin_size}_thr{threshold_size}.png"
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved as: {save_path}")
    
    plt.show()
    
    # Print percentage breakdown with proportions
    print(f"\nRandom Forest Percentage breakdown by actual label (Bin={bin_size}, Threshold={threshold_size}):")
    for i, actual_label in enumerate(class_labels):
        row_total = cm.sum(axis=1)[i]
        print(f"\n{actual_label} (Total: {row_total} samples):")
        for j, pred_label in enumerate(class_labels):
            count = cm[i, j]
            percentage = cm_percent[i, j]
            print(f"  → {pred_label}: {percentage:.1f}% ({count}/{row_total} samples)")
    
    return cm_percent, df

In [ ]:
# Example usage for Random Forest Confusion Matrix
# Replace with your actual folder path, folder name, and parameter values
cm_percent_rf, df_rf = create_percentage_confusion_matrix_random_forest(
    folder="/home/dlipsey/MITLincolnLabs/MIT_LL_data/",
    folder_name="random_forest_df6",  # Update with your RF results folder name
    bin_size=1,
    threshold_size=0.05,
    save_plots=False
)

In [ ]:
# Create percentage-based confusion matrix heatmap for Random Forest results
def create_percentage_confusion_matrix_random_forest(folder, folder_name, bin_size, threshold_size, save_plots=True, test_only=False):
    """
    Standalone function to create percentage confusion matrix from random forest classification outputs
    Each row shows how the actual label was distributed across predictions
    
    Parameters:
    test_only (bool): If True, only include test/validation spectra (exclude training spectra where train=1)
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import confusion_matrix
    import os
  
    def response_to_tox_class_local(response_value):
        """Convert Response value to Tox toxicity class"""
        if response_value <= 5:
            return 0  # Tox Level 0 
        elif response_value <= 50:
            return 1  # Tox Level 1
        elif response_value <= 500:
            return 2  # Tox Level 2
        elif response_value <= 5000:
            return 3  # Tox Level 3
        else:
            return 4  # Tox Level 4
    
    # Load the random forest classification outputs
    folder_path = os.path.join(folder, folder_name)
    
    # Convert bin_size and threshold_size to the naming format (replace . with _)
    bin_part = str(bin_size).replace('.', '_')
    threshold_part = str(threshold_size).replace('.', '_')
    
    # Random forest file naming pattern
    filename = f"super_test_rf_bin{bin_part}_thresh{threshold_part}_df_spectra.parquet"
    
    outputs_file = os.path.join(folder_path, filename)
    
    if not os.path.exists(outputs_file):
        raise FileNotFoundError(f"Output file not found: {outputs_file}")
    
    df = pd.read_parquet(outputs_file)
    
    # Filter to test-only spectra if requested
    if test_only:
        if 'train' in df.columns:
            original_count = len(df)
            df = df[df['train'] == 0].copy()
            filtered_count = len(df)
            print(f"Filtered to test-only spectra: {filtered_count}/{original_count} samples ({filtered_count/original_count*100:.1f}%)")
        else:
            print("Warning: 'train' column not found, cannot filter to test-only spectra")
    
    if not os.path.exists(outputs_file):
        raise FileNotFoundError(f"Output file not found: {outputs_file}")
    
    df = pd.read_parquet(outputs_file)
    
    # Get true labels - prioritize Tox_level if available, otherwise derive from Response
    if 'Tox_level' in df.columns:
        y_true_raw = df['Tox_level'].values  
    else:
        # Derive Tox class from Response values
        df['Tox_class'] = df['Response'].apply(response_to_tox_class_local)
        y_true_raw = df['Tox_class'].values  
    
    # Get predicted classes from random forest
    y_pred_raw = df['rf_predicted_tox_class'].values
    
    # Convert both to 0-based indexing for sklearn
    y_true = y_true_raw  
    y_pred = y_pred_raw  
    
    # Define Tox labels
    class_labels = ["Tox Level 0", "Tox Level 1", "Tox Level 2", "Tox Level 3", "Tox Level 4"]
    
    # Create confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])
    
    # Convert to percentages (row-wise normalization)
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Create the plot
    plt.figure(figsize=(10, 8))
    
    # Create heatmap with percentage values
    sns.heatmap(cm_percent, 
                annot=True, 
                fmt='.1f', 
                cmap='Blues', 
                xticklabels=class_labels, 
                yticklabels=class_labels,
                cbar_kws={'label': 'Percentage (%)'},
                square=True)
    
    test_label = " (Test Only)" if test_only else ""
    plt.title(f"Random Forest Confusion Matrix - Percentages{test_label}\n(Bin={bin_size}, Threshold={threshold_size})", 
              fontsize=14, fontweight='bold')
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('Actual', fontsize=12)
    plt.tight_layout()
    
    if save_plots:
        save_path = f"confusion_matrix_percentages_rf_bin{bin_size}_thr{threshold_size}.png"
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved as: {save_path}")
    
    plt.show()
    
    # Print percentage breakdown with proportions
    print(f"\nRandom Forest Percentage breakdown by actual label (Bin={bin_size}, Threshold={threshold_size}):")
    for i, actual_label in enumerate(class_labels):
        row_total = cm.sum(axis=1)[i]
        print(f"\n{actual_label} (Total: {row_total} samples):")
        for j, pred_label in enumerate(class_labels):
            count = cm[i, j]
            percentage = cm_percent[i, j]
            print(f"  → {pred_label}: {percentage:.1f}% ({count}/{row_total} samples)")
    
    return cm_percent, df

In [ ]:
# Example usage for Random Forest Confusion Matrix
# Super test
cm_percent_rf, df_rf = create_percentage_confusion_matrix_random_forest(
    folder="/home/dlipsey/MITLincolnLabs/MIT_LL_data/",
    folder_name="random_forest_df6_super_test",  # Update with your RF results folder name
    bin_size=1,
    threshold_size=0.05,
    save_plots=False
)

## Heatmap

In [ ]:
# Configurable Accuracy Heatmap Function for Random Forest
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, accuracy_score

def response_ttox_class_local_rf(response_value):
    """Convert Response value to tox toxicity class"""
    if response_value <= 5:
        return 0  # Tox Level 0
    elif response_value <= 50:
        return 1  # Tox Level 1
    elif response_value <= 500:
        return 2  # Tox Level 2
    elif response_value <= 5000:
        return 3  # Tox Level 3
    else:
        return 4  # Tox Level 4

def calculate_class_accuracy_rf(y_true, y_pred, target_class):
    """Calculate accuracy for a specific class"""
    # Convert to 0-based indexing
    target_class_idx = target_class - 1
    
    # Get mask for samples that actually belong to target class
    class_mask = y_true == target_class_idx
    
    if np.sum(class_mask) == 0:
        return np.nan  # No samples of this class
    
    # Calculate accuracy for this class (correct predictions / total actual samples of this class)
    correct_for_class = np.sum((y_true == y_pred) & class_mask)
    total_for_class = np.sum(class_mask)
    
    return correct_for_class / total_for_class

def parse_rf_classification_dataset_name(dataset_name):
    """Extract bin size and threshold from random forest classification dataset name"""
    # Remove 'rf_' prefix
    name_part = dataset_name.replace('rf_', '') 
    
    # Handle thresh_zero case (no threshold)
    if 'thresh_zero' in name_part:
        # Extract bin size
        bin_part = name_part.split('_thresh_zero')[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        threshold = 0.0
    else:
        # Extract bin size and threshold
        parts = name_part.split('_thresh')
        bin_part = parts[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        
        thresh_part = parts[1].split('_df_spectra')[0]
        threshold = float(thresh_part.replace('_', '.'))
    
    return bin_size, threshold

def load_and_process_rf_classification_data(folder_path):
    """Load and process random forest classification data from the specified folder"""
    # Check if folder exists
    if not os.path.exists(folder_path):
        print(f"Error: Folder {folder_path} does not exist!")
        return pd.DataFrame()
    
    # Get all .parquet files in the folder
    parquet_files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]
    dataset_names = [f.replace('.parquet', '') for f in parquet_files]

    if not dataset_names:
        print(f"Warning: No .parquet files found in {folder_path}")
        return pd.DataFrame()

    # Initialize storage for classification results
    classification_results = []

    # Process classification datasets
    for i, dataset_name in enumerate(sorted(dataset_names), 1):
        try:
            # Load the classification dataset
            file_path = os.path.join(folder_path, f"{dataset_name}.parquet")
            df = pd.read_parquet(file_path)
            
            # Check if required columns exist
            if 'rf_predicted_tox_class' not in df.columns:
                print(f"  Warning: 'rf_predicted_tox_class' column not found, skipping...")
                continue
            
            # Get predictions and true values
            y_pred_class = df['rf_predicted_tox_class']  # Predicted Tox classes
            
            # Get true labels - prioritize Tox_level if available, otherwise derive from Response
            if 'Tox_level' in df.columns:
                y_true_class = df['Tox_level']  # Should be 0,1,2,3,4
            elif 'Response' in df.columns:
                y_true_class = df['Response'].apply(response_to_tox_class_local_rf)
            else:
                print(f"  Warning: Neither 'Tox_level' nor 'Response' column found, skipping...")
                continue
            
            # Remove rows with NaN values
            valid_mask = ~(y_pred_class.isna() | y_true_class.isna())
            y_pred_class_clean = y_pred_class[valid_mask]
            y_true_class_clean = y_true_class[valid_mask]
            
            if len(y_pred_class_clean) < 10:  # Skip if too few samples
                print(f"  Too few samples, skipping...")
                continue
            
            # Convert to 0-based indexing for sklearn functions
            y_true_class_clean_0based = y_true_class_clean
            y_pred_class_clean_0based = y_pred_class_clean
            
            # Calculate overall accuracy
            overall_accuracy = accuracy_score(y_true_class_clean_0based, y_pred_class_clean_0based)
            
            # Calculate individual class accuracies
            class1_accuracy = calculate_class_accuracy_rf(y_true_class_clean_0based, y_pred_class_clean_0based, 1)
            class2_accuracy = calculate_class_accuracy_rf(y_true_class_clean_0based, y_pred_class_clean_0based, 2)
            class3_accuracy = calculate_class_accuracy_rf(y_true_class_clean_0based, y_pred_class_clean_0based, 3)
            class4_accuracy = calculate_class_accuracy_rf(y_true_class_clean_0based, y_pred_class_clean_0based, 4)
            
            # Store results
            classification_results.append({
                'Dataset': dataset_name,
                'Overall_Accuracy': overall_accuracy,
                'Class1_Accuracy': class1_accuracy,
                'Class2_Accuracy': class2_accuracy,
                'Class3_Accuracy': class3_accuracy,
                'Class4_Accuracy': class4_accuracy,
                'Samples': len(y_pred_class_clean)
            })
            
        except Exception as e:
            print(f"  ✗ Error processing {dataset_name}: {str(e)}")
            continue

    # Convert results to DataFrame
    results_df = pd.DataFrame(classification_results)

    if results_df.empty:
        print("Warning: No datasets were successfully processed!")
        return results_df

    # Add bin_size and threshold columns
    bin_sizes = []
    thresholds = []

    for dataset_name in results_df['Dataset']:
        bin_size, threshold = parse_rf_classification_dataset_name(dataset_name)
        bin_sizes.append(bin_size)
        thresholds.append(threshold)

    results_df['BinSize'] = bin_sizes
    results_df['Threshold'] = thresholds

    # Remove duplicates
    results_df = results_df.drop_duplicates(subset=['BinSize', 'Threshold'], keep='first')
    
    return results_df

# Main function to create accuracy heatmap with configurable parameters for Random Forest
def create_detailed_heatmap_rf_accuracy(folder_path, metric='average', colormap='RdYlGn', figsize=(12, 8)):
    """
    Create a detailed heatmap for Random Forest Classification Accuracy
    
    Parameters:
    folder_path (str): Path to folder containing classification result files
    metric (str): Type of accuracy to plot
        - 'average': Overall accuracy
        - '0', '1', '2', '3', '4': Individual class accuracy (Tox Level 0-4)
    colormap (str): Color scheme for the heatmap (e.g., 'RdYlGn', 'viridis', 'Blues', 'Reds', 'Greens', 'Purples')
    figsize (tuple): Figure size as (width, height)
    
    Returns:
    pandas.DataFrame: Pivot table with accuracy data, or None if no data available
    """
    
    # Load and process data from the specified folder
    df_results = load_and_process_rf_classification_data(folder_path)
    
    if df_results.empty:
        print("No data available to create heatmap!")
        print("Please ensure the folder contains valid .parquet files with required columns.")
        return None
    
    # Map metric types to column names and display names
    metric_mapping = {
        'average': ('Overall_Accuracy', 'Overall Accuracy'),
        '0': ('Class0_Accuracy', 'Tox Level 0 Accuracy'),
        '1': ('Class1_Accuracy', 'Tox Level 1 Accuracy'),
        '2': ('Class2_Accuracy', 'Tox Level 2 Accuracy'),
        '3': ('Class3_Accuracy', 'Tox Level 3 Accuracy'),
        '4': ('Class4_Accuracy', 'Tox Level 4 Accuracy')
    }
    
    if metric not in metric_mapping:
        raise ValueError(f"metric must be one of: {list(metric_mapping.keys())}")
    
    column_name, display_name = metric_mapping[metric]
    
    # Create pivot table
    accuracy_pivot = df_results.pivot(index='BinSize', columns='Threshold', values=column_name)
    
    # List all expected thresholds and bin sizes
    thresholds_subset = [0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 50, 100]
    bins_subset = [0.1, 0.5, 1, 2, 5, 10, 25, 50, 100, 200]
    
    # Reindex pivot table
    accuracy_pivot = accuracy_pivot.reindex(columns=thresholds_subset, index=bins_subset)
    
    plt.figure(figsize=figsize)
    
    # Create heatmap
    ax = sns.heatmap(accuracy_pivot, 
                     annot=True, 
                     fmt='.3f', 
                     cmap=colormap,
                     square=False,
                     linewidths=0.5,
                     vmin=0,
                     vmax=1,
                     cbar_kws={'label': display_name, 'shrink': 0.8})
    
    plt.title(f'Random Forest Results: {display_name} by Bin Size and Threshold', fontsize=16, fontweight='bold')
    plt.xlabel('Threshold Value', fontsize=14)
    plt.ylabel('Bin Size', fontsize=14)
    plt.gca().invert_yaxis()
    
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    
    # Add text annotation for best performance
    best_acc = accuracy_pivot.max().max()
    plt.text(0.02, 0.98, f'Best {display_name}: {best_acc:.3f}', 
            transform=plt.gca().transAxes, 
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
            verticalalignment='top')
    
    plt.tight_layout()
    plt.show()
    
    return accuracy_pivot

# Define the folder path for Random Forest results
folder_path_rf = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/random_forest_df6"

# Overall accuracy with default green color scheme
create_detailed_heatmap_rf_accuracy(folder_path_rf, metric='average', colormap='RdYlGn')

# Old Version saved for posterity

In [ ]:
# Configurable Accuracy Heatmap Function
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, accuracy_score

def response_to_tox_class_local(response_value):
    """Convert Response value to Tox toxicity class"""
    if response_value <= 5: 
        return 0  # Tox Level 0
    elif response_value <= 50:
        return 1  # Tox Level 1
    elif response_value <= 500:
        return 2  # Tox Level 2
    elif response_value <= 5000:
        return 3  # Tox Level 3
    else:
        return 4  # Tox Level 4

def calculate_class_accuracy(y_true, y_pred, target_class):
    """Calculate accuracy for a specific class"""
    # Convert to 0-based indexing
    target_class_idx = target_class - 1
    
    # Get mask for samples that actually belong to target class
    class_mask = y_true == target_class_idx
    
    if np.sum(class_mask) == 0:
        return np.nan  # No samples of this class
    
    # Calculate accuracy for this class (correct predictions / total actual samples of this class)
    correct_for_class = np.sum((y_true == y_pred) & class_mask)
    total_for_class = np.sum(class_mask)
    
    return correct_for_class / total_for_class

def parse_cond_enc_classification_dataset_name(dataset_name):
    """Extract bin size and threshold from full conditional encoder classification dataset name"""
    # Remove 'cond_enc_' prefix
    name_part = dataset_name.replace('cond_enc_', '') 
    
    # Handle thresh_zero case (no threshold)
    if 'thresh_zero' in name_part:
        # Extract bin size
        bin_part = name_part.split('_thresh_zero')[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        threshold = 0.0
    else:
        # Extract bin size and threshold
        parts = name_part.split('_thresh')
        bin_part = parts[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        
        thresh_part = parts[1].split('_df_spectra')[0]
        threshold = float(thresh_part.replace('_', '.'))
    
    return bin_size, threshold

def load_and_process_classification_data(folder_path):
    """Load and process classification data from the specified folder"""
    # Check if folder exists
    if not os.path.exists(folder_path):
        print(f"Error: Folder {folder_path} does not exist!")
        return pd.DataFrame()
    
    # Get all .parquet files in the folder
    parquet_files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]
    dataset_names = [f.replace('.parquet', '') for f in parquet_files]

    if not dataset_names:
        print(f"Warning: No .parquet files found in {folder_path}")
        return pd.DataFrame()

    # Initialize storage for classification results
    classification_results = []

    # Process classification datasets
    for i, dataset_name in enumerate(sorted(dataset_names), 1):
        try:
            # Load the classification dataset
            file_path = os.path.join(folder_path, f"{dataset_name}.parquet")
            df = pd.read_parquet(file_path)
            
            # Check if required columns exist
            if 'cond_tox_pred_class' not in df.columns:
                print(f"  Warning: 'cond_tox_pred_class' column not found, skipping...")
                continue
            
            if 'Response' not in df.columns:
                print(f"  Warning: 'Response' column not found, skipping...")
                continue
            
            # Get predictions and true values
            y_pred_class = df['cond_tox_pred_class']  # Predicted Tox classes
            y_true_response = df['Response']  # True toxicity values
            
            # Convert true response values to Tox classes
            y_true_class = y_true_response.apply(response_to_tox_class_local)
            
            # Remove rows with NaN values
            valid_mask = ~(y_pred_class.isna() | y_true_class.isna())
            y_pred_class_clean = y_pred_class[valid_mask]
            y_true_class_clean = y_true_class[valid_mask]
            
            if len(y_pred_class_clean) < 10:  # Skip if too few samples
                print(f"  Too few samples, skipping...")
                continue
            
            # Convert to 0-based indexing for sklearn functions
            y_true_class_clean_0based = y_true_class_clean - 1
            
            # Calculate overall accuracy
            overall_accuracy = accuracy_score(y_true_class_clean_0based, y_pred_class_clean)
            
            # Calculate individual class accuracies
            class1_accuracy = calculate_class_accuracy(y_true_class_clean_0based, y_pred_class_clean, 1)
            class2_accuracy = calculate_class_accuracy(y_true_class_clean_0based, y_pred_class_clean, 2)
            class3_accuracy = calculate_class_accuracy(y_true_class_clean_0based, y_pred_class_clean, 3)
            class4_accuracy = calculate_class_accuracy(y_true_class_clean_0based, y_pred_class_clean, 4)
            
            # Store results
            classification_results.append({
                'Dataset': dataset_name,
                'Overall_Accuracy': overall_accuracy,
                'Class1_Accuracy': class1_accuracy,
                'Class2_Accuracy': class2_accuracy,
                'Class3_Accuracy': class3_accuracy,
                'Class4_Accuracy': class4_accuracy,
                'Samples': len(y_pred_class_clean)
            })
            
        except Exception as e:
            print(f"  ✗ Error processing {dataset_name}: {str(e)}")
            continue

    # Convert results to DataFrame
    results_df = pd.DataFrame(classification_results)

    if results_df.empty:
        print("Warning: No datasets were successfully processed!")
        return results_df

    # Add bin_size and threshold columns
    bin_sizes = []
    thresholds = []

    for dataset_name in results_df['Dataset']:
        bin_size, threshold = parse_cond_enc_classification_dataset_name(dataset_name)
        bin_sizes.append(bin_size)
        thresholds.append(threshold)

    results_df['BinSize'] = bin_sizes
    results_df['Threshold'] = thresholds

    # Remove duplicates
    results_df = results_df.drop_duplicates(subset=['BinSize', 'Threshold'], keep='first')
    
    return results_df

# Main function to create accuracy heatmap with configurable parameters
def create_detailed_heatmap_cond_enc_accuracy(folder_path, metric='average', colormap='RdYlGn', figsize=(12, 8)):
    """
    Create a detailed heatmap for Classification Accuracy
    
    Parameters:
    folder_path (str): Path to folder containing classification result files
    metric (str): Type of accuracy to plot
        - 'average': Overall accuracy
        - '0', '1', '2', '3', '4': Individual class accuracy (Tox Level 0-4)
    colormap (str): Color scheme for the heatmap (e.g., 'RdYlGn', 'viridis', 'Blues', 'Reds', 'Greens', 'Purples')
    figsize (tuple): Figure size as (width, height)
    
    Returns:
    pandas.DataFrame: Pivot table with accuracy data, or None if no data available
    """
    
    # Load and process data from the specified folder
    df_results = load_and_process_classification_data(folder_path)
    
    if df_results.empty:
        print("No data available to create heatmap!")
        print("Please ensure the folder contains valid .parquet files with required columns.")
        return None
    
    # Map metric types to column names and display names
    metric_mapping = {
        'average': ('Overall_Accuracy', 'Overall Accuracy'),
        '0': ('Class0_Accuracy', 'Tox Level 0 Accuracy'),
        '1': ('Class1_Accuracy', 'Tox Level 1 Accuracy'),
        '2': ('Class2_Accuracy', 'Tox Level 2 Accuracy'),
        '3': ('Class3_Accuracy', 'Tox Level 3 Accuracy'),
        '4': ('Class4_Accuracy', 'Tox Level 4 Accuracy')
    }
    
    if metric not in metric_mapping:
        raise ValueError(f"metric must be one of: {list(metric_mapping.keys())}")
    
    column_name, display_name = metric_mapping[metric]
    
    # Create pivot table
    accuracy_pivot = df_results.pivot(index='BinSize', columns='Threshold', values=column_name)
    
    # List all expected thresholds and bin sizes
    thresholds_subset = [0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 50, 100]
    bins_subset = [0.1, 0.5, 1, 2, 5, 10, 25, 50, 100, 200, 500]
    
    # Reindex pivot table
    accuracy_pivot = accuracy_pivot.reindex(columns=thresholds_subset, index=bins_subset)
    
    plt.figure(figsize=figsize)
    
    # Create heatmap
    ax = sns.heatmap(accuracy_pivot, 
                     annot=True, 
                     fmt='.3f', 
                     cmap=colormap,
                     square=False,
                     linewidths=0.5,
                     vmin=0,
                     vmax=1,
                     cbar_kws={'label': display_name, 'shrink': 0.8})
    
    # Rectangle highlighting has been removed
    
    plt.title(f'Classification Results: {display_name} by Bin Size and Threshold', fontsize=16, fontweight='bold')
    plt.xlabel('Threshold Value', fontsize=14)
    plt.ylabel('Bin Size', fontsize=14)
    plt.gca().invert_yaxis()
    
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    
    # Add text annotation for best performance
    best_acc = accuracy_pivot.max().max()
    plt.text(0.02, 0.98, f'Best {display_name}: {best_acc:.3f}', 
            transform=plt.gca().transAxes, 
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
            verticalalignment='top')
    
    plt.tight_layout()
    plt.show()
    
    return accuracy_pivot
